# Anchor-free: A1 — YOLO11s, imgsz 1280
Той самий ноутбук і протокол, що A0; змінено лише роздільну здатність входу 1024 → 1280. Повний лог тренування — у W&B.

In [1]:
RUN_ID   = "A1_yolo11s_1280"
FAMILY   = "anchor_free"
MODEL    = "yolo11s.pt"
CHANGE   = "imgsz 1280"
IMGSZ    = 1280
EPOCHS   = 10
BATCH    = 16                      # сумарно на 2 GPU
TRAIN_KW = dict()                  # додаткові аргументи model.train(...) для тюнінгу
DEBUG    = False                    # True: 2 кліпи, 1 епоха, 50 кадрів — перевірка за ~5 хв

PROTOCOL = dict(train_step=4, val_step=2, train_max_side=1600, seed=0,
                conf=0.001, nms_iou=0.7, max_det=500)
NAMES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
WANDB_PROJECT = "hw2-ucu-2026"
WANDB_ENTITY  = "solchanykv-ukrainian-catholic-university"

import importlib.util, os, subprocess, sys
for pkg in ["ultralytics", "pycocotools"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import contextlib, copy, gc, io, json, pickle, random, shutil, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO, RTDETR, settings


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def find_root():
    for pat in ["*", "*/*", "*/*/*"]:
        for c in Path("/kaggle/input").glob(pat):
            if (c / "train" / "annotations_coco.json").exists():
                return c
    raise FileNotFoundError("Додай дані змагання в Input")


def wandb_login():
    import wandb
    try:
        from kaggle_secrets import UserSecretsClient; key = UserSecretsClient().get_secret("hw2")
    except Exception as e:
        print("W&B вимкнено:", e)
        return False
    os.environ.update(WANDB_API_KEY=key, WANDB_PROJECT=WANDB_PROJECT, WANDB_ENTITY=WANDB_ENTITY)
    return wandb.login(key=key, relogin=True)


seed_everything(PROTOCOL["seed"])
ROOT = find_root()
WORK = Path("/kaggle/working"); DS = WORK / "ds"; ART = WORK / "artifacts"; ART.mkdir(exist_ok=True)
MAX_CLIPS = 2 if DEBUG else None
if DEBUG:
    EPOCHS = 1

assert torch.cuda.device_count() == 2, "Accelerator: GPU T4 x2"
USE_WANDB = wandb_login()
settings.update({"wandb": USE_WANDB})
print(f"дані: {ROOT}\nultralytics {ultralytics.__version__} | torch {torch.__version__} | "
      f"GPU: {torch.cuda.get_device_name(0)} x{torch.cuda.device_count()} | W&B: {USE_WANDB}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: solchanykv (solchanykv-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
дані: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
ultralytics 8.4.213 | torch 2.11.0+cu130 | GPU: Tesla T

In [2]:
GRAY = 114


def frame_key(path):
    p = Path(path)
    return p.parent.name, p.stem


def read_yolo(path, W, H):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return np.zeros((0, 5))
    a = np.loadtxt(p, ndmin=2); out = a.copy()
    out[:, 1] = (a[:, 1] - a[:, 3] / 2) * W; out[:, 2] = (a[:, 2] - a[:, 4] / 2) * H
    out[:, 3] = a[:, 3] * W;                 out[:, 4] = a[:, 4] * H
    return out                                          # cls, x, y, w, h (px)


def read_ignore(path, W, H):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return np.zeros((0, 4))
    a = np.loadtxt(p, ndmin=2)[:, -4:]
    return np.c_[(a[:, 0] - a[:, 2] / 2) * W, (a[:, 1] - a[:, 3] / 2) * H, a[:, 2] * W, a[:, 3] * H]


def frac_inside(boxes, regions):
    if len(boxes) == 0 or len(regions) == 0:
        return np.zeros(len(boxes))
    a = np.c_[boxes[:, :2], boxes[:, :2] + boxes[:, 2:4]]
    b = np.c_[regions[:, :2], regions[:, :2] + regions[:, 2:4]]
    iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    return (iw * ih / np.maximum(boxes[:, 2] * boxes[:, 3], 1e-9)[:, None]).max(1)


def list_frames(split, step=1, max_clips=None):
    clips = sorted(p for p in (ROOT / split / "images").iterdir() if p.is_dir())
    return [f for c in clips[:max_clips] for f in sorted(c.glob("*.jpg"))[::step]]


def _process(f, split, mask, max_side, drop_thr):
    clip, stem = f.parent.name, f.stem
    dst = DS / split / "images" / clip / f.name
    lbl = ROOT / split / "labels" / clip / f"{stem}.txt"
    ign = ROOT / split / "ignore" / clip / f"{stem}.txt"
    lines = lbl.read_text().strip().splitlines() if lbl.exists() else []
    n_drop = 0
    need_mask = mask and ign.exists() and ign.stat().st_size > 0
    if need_mask or max_side:
        img = cv2.imread(str(f)); H, W = img.shape[:2]
        if need_mask:
            reg = read_ignore(ign, W, H)
            for x, y, w, h in reg:
                img[max(0, int(y)):int(np.ceil(y + h)), max(0, int(x)):int(np.ceil(x + w))] = GRAY
            if lines:
                keep = frac_inside(read_yolo(lbl, W, H)[:, 1:], reg) <= drop_thr
                n_drop = int((~keep).sum()); lines = [l for l, k in zip(lines, keep) if k]
        if max_side and max(H, W) > max_side:
            s = max_side / max(H, W)
            img = cv2.resize(img, (round(W * s), round(H * s)), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(dst), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    elif not dst.exists():
        dst.symlink_to(f)
    (DS / split / "labels" / clip / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
    return n_drop


def build_split(split, step, mask=False, max_side=None, drop_thr=0.5, max_clips=None):
    frames = list_frames(split, step, max_clips)
    for clip in {f.parent.name for f in frames}:
        (DS / split / "images" / clip).mkdir(parents=True, exist_ok=True)
        (DS / split / "labels" / clip).mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(8) as ex:
        drops = list(ex.map(lambda f: _process(f, split, mask, max_side, drop_thr), frames))
    print(f"{split}: {len(frames)} кадрів, видалено GT під ignore: {sum(drops)}")


if not (DS / "data.yaml").exists():
    t0 = time.time()
    build_split("train", PROTOCOL["train_step"], mask=True, max_side=PROTOCOL["train_max_side"], max_clips=MAX_CLIPS)
    build_split("val", PROTOCOL["val_step"], mask=False, max_clips=MAX_CLIPS)
    (DS / "data.yaml").write_text(f"path: {DS}\ntrain: train/images\nval: val/images\nnames:\n"
                                  + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
    print(f"зібрано за {time.time() - t0:.0f} с")

VAL = list_frames("val", PROTOCOL["val_step"], MAX_CLIPS)
test_df = pd.read_csv(ROOT / "test_images.csv")
TEST = [ROOT / f for f in test_df.file_name]
if DEBUG:
    VAL, TEST = VAL[:50], TEST[:50]
print(f"val: {len(VAL)} | test: {len(TEST)}")

train: 6072 кадрів, видалено GT під ignore: 3069
val: 1425 кадрів, видалено GT під ignore: 0
зібрано за 122 с
val: 1425 | test: 3811


In [3]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


class Evaluator:
    """COCOeval по кадрах VAL: ignore через iscrowd=1, не більше 500 детекцій на кадр, AP по класах."""

    def __init__(self, gt_json, image_paths):
        with contextlib.redirect_stdout(io.StringIO()):
            self.gt = COCO(str(gt_json))
        self.key2id = {frame_key(i["file_name"]): i["id"] for i in self.gt.dataset["images"]}
        self.keys = [frame_key(p) for p in image_paths]
        self.img_ids = [self.key2id[k] for k in self.keys]

    def to_coco(self, preds):
        out = []
        for k, (xyxy, s, c) in preds.items():
            if k not in self.key2id:
                continue
            for i in np.argsort(-np.asarray(s, np.float32))[:PROTOCOL["max_det"]]:
                x1, y1, x2, y2 = map(float, xyxy[i])
                out.append(dict(image_id=self.key2id[k], category_id=int(c[i]) + 1,
                                bbox=[x1, y1, x2 - x1, y2 - y1], score=float(s[i])))
        return out

    def evaluate(self, preds):
        dets = self.to_coco(preds)
        if not dets:
            return dict(mAP=0.0)
        with contextlib.redirect_stdout(io.StringIO()):
            E = COCOeval(self.gt, self.gt.loadRes(dets), "bbox")
            E.params.imgIds = self.img_ids
            E.params.maxDets = [1, 100, PROTOCOL["max_det"]]
            E.evaluate(); E.accumulate(); E.summarize()
        s, prec = E.stats, E.eval["precision"]          # precision: [IoU, recall, class, area, maxDets]
        p_all = prec[:, :, :, 0, -1]                     # pycocotools рахує stats[0] при maxDets=100 — беремо 500
        m = dict(mAP=p_all[p_all > -1].mean(), AP50=s[1], AP75=s[2], AP_small=s[3], AP_medium=s[4],
                 AP_large=s[5], AR500=s[8])
        for k, n in enumerate(NAMES):
            p = prec[:, :, k, 0, -1]
            m[f"AP_{n}"] = p[p > -1].mean() if (p > -1).any() else np.nan
        return {k: float(v) for k, v in m.items()}


ev = Evaluator(ROOT / "val" / "annotations_coco.json", VAL)
gt_preds = {}
for k, iid in zip(ev.keys, ev.img_ids):
    anns = [a for a in ev.gt.imgToAnns[iid] if not a.get("iscrowd", 0)]
    b = np.array([a["bbox"] for a in anns], np.float32).reshape(-1, 4); b[:, 2:] += b[:, :2]
    gt_preds[k] = (b, np.ones(len(b), np.float32), np.array([a["category_id"] - 1 for a in anns], np.int8))
sanity = ev.evaluate(gt_preds)
print({k: round(v, 4) for k, v in sanity.items() if k in ("mAP", "AP50", "AP_small")})
assert sanity["mAP"] > 0.99, "оцінка рахує неправильно"
del gt_preds

{'mAP': 1.0, 'AP50': 1.0, 'AP_small': 1.0}


In [4]:
Model = RTDETR if "rtdetr" in MODEL else YOLO
model = Model(MODEL)
model.train(data=str(DS / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=[0, 1],
            workers=4, cache=False, seed=PROTOCOL["seed"], deterministic=True, cos_lr=True,
            close_mosaic=2, amp=True, project=WANDB_PROJECT, name=RUN_ID, exist_ok=True, plots=True,
            **TRAIN_KW)
del model; gc.collect(); torch.cuda.empty_cache()

run_dir = max((p.parent.parent for p in WORK.rglob("weights/best.pt") if p.parent.parent.name == RUN_ID),
              key=lambda d: d.stat().st_mtime)
OUT = ART / RUN_ID; OUT.mkdir(exist_ok=True)
shutil.copy(run_dir / "weights" / "best.pt", OUT / "best.pt")
for f in ["results.csv", "args.yaml", "results.png", "confusion_matrix_normalized.png"]:
    if (run_dir / f).exists():
        shutil.copy(run_dir / f, OUT / f)
print("тренування:", run_dir, "\nваги:", OUT / "best.pt")

Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=2, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/ds/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.

In [5]:
import logging
from ultralytics.utils import LOGGER
LOGGER.setLevel(logging.ERROR)


def predict_ultra(model, paths, imgsz, batch=8, chunk=64):
    preds = {}
    paths = [str(p) for p in paths]
    for s in range(0, len(paths), chunk):
        for r in model.predict(source=paths[s:s + chunk], imgsz=imgsz, conf=PROTOCOL["conf"],
                               iou=PROTOCOL["nms_iou"], max_det=PROTOCOL["max_det"], half=True,
                               batch=batch, stream=True, verbose=False, device=0):
            b = r.boxes
            preds[frame_key(r.path)] = (b.xyxy.cpu().numpy().astype(np.float32),
                                        b.conf.cpu().numpy().astype(np.float32),
                                        b.cls.cpu().numpy().astype(np.int8))
        gc.collect(); torch.cuda.empty_cache()
    return preds


def measure_fps(predict_fn, paths, n=100, warmup=10):
    paths = list(paths)[: n + warmup]
    for p in paths[:warmup]:
        predict_fn(p)
    torch.cuda.synchronize(); t = time.perf_counter()
    for p in paths[warmup:]:
        predict_fn(p)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t) / max(1, len(paths) - warmup) * 1000
    return dict(ms_per_img=ms, FPS=1000 / ms)


model = Model(str(OUT / "best.pt"))
val_preds = predict_ultra(model, VAL, IMGSZ)
metrics = ev.evaluate(val_preds)
metrics.update(measure_fps(lambda p: model.predict(p, imgsz=IMGSZ, half=True, conf=PROTOCOL["conf"],
                                                   max_det=PROTOCOL["max_det"], verbose=False, device=0), VAL))
main = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500", "FPS"]
per_class = pd.Series({n: metrics[f"AP_{n}"] for n in NAMES}, name="AP")
display(pd.Series({k: metrics[k] for k in main}).round(4).to_frame(RUN_ID).T)
display(per_class.round(4).to_frame().T)

,mAP,AP50,AP75,AP_small,AP_medium,AP_large,AR500,FPS
A1_yolo11s_1280,0.1908,0.3787,0.1692,0.111,0.1938,0.2892,0.3177,48.4149


,person,bicycle,car,van,truck,tricycle,bus,motor
AP,0.288,0.1735,0.404,0.1433,0.2125,0.1765,0.013,0.116


In [6]:
def boxes_image(path, pred, conf=0.3, max_side=1280):
    import wandb
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    s = min(1.0, max_side / max(H, W))
    if s < 1:
        img = cv2.resize(img, (round(W * s), round(H * s)), interpolation=cv2.INTER_AREA)
    clip, stem = frame_key(path)
    cl = dict(enumerate(NAMES))
    gt = read_yolo(ROOT / "val" / "labels" / clip / f"{stem}.txt", W, H)
    xyxy, sc, c = pred
    keep = sc >= conf
    box = lambda x1, y1, x2, y2: {"minX": float(x1) * s, "minY": float(y1) * s, "maxX": float(x2) * s, "maxY": float(y2) * s}
    boxes = {
        "ground_truth": {"class_labels": cl, "box_data": [
            {"position": box(x, y, x + w, y + h), "domain": "pixel", "class_id": int(k)} for k, x, y, w, h in gt]},
        "predictions": {"class_labels": cl, "box_data": [
            {"position": box(*b), "domain": "pixel", "class_id": int(k), "scores": {"conf": float(v)}}
            for b, v, k in zip(xyxy[keep], sc[keep], c[keep])]},
    }
    return wandb.Image(img, boxes=boxes, caption=f"{clip}/{stem}")


if USE_WANDB:
    import wandb
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=f"{RUN_ID}_eval", group=FAMILY,
                     job_type="eval", config=dict(PROTOCOL, run_id=RUN_ID, model=MODEL, change=CHANGE,
                                                  imgsz=IMGSZ, epochs=EPOCHS, debug=DEBUG, **TRAIN_KW))
    run.summary.update(metrics)
    run.log({"per_class_AP": wandb.Table(dataframe=per_class.reset_index().rename(columns={"index": "class"}))})
    idx = np.linspace(0, len(VAL) - 1, 12).astype(int)
    run.log({"val_predictions": [boxes_image(VAL[i], val_preds[frame_key(VAL[i])]) for i in idx]})
    run.finish()

wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261115_001932-raxqaqy9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run A1_yolo11s_1280_eval
wandb: ⭐️ View project at https://wandb.ai/solchanykv-ukrainian-catholic-university/hw2-ucu-2026
wandb: 🚀 View run at https://wandb.ai/solchanykv-ukrainian-catholic-university/hw2-ucu-2026/runs/raxqaqy9
wandb: updating run metadata
wandb: 
wandb: Run summary:
wandb:       AP50 0.37868
wandb:       AP75 0.16924
wandb: AP_bicycle 0.17348
wandb:     AP_bus 0.01301
wandb:     AP_car 0.40395
wandb:   AP_large 0.28921
wandb:  AP_medium 0.19379
wandb:   AP_motor 0.11601
wandb:  AP_person 0.28796
wandb:   AP_small 0.11102
wandb:         +7 ...
wandb: 
wandb: 🚀 View run A1_yolo11s_1280_eval at: https://wandb.ai/solchanykv-ukrainian-catholic-university/hw2-ucu-2026/runs/raxqaqy9
wandb: ⭐️ View project at: https://wandb.ai/solchanykv-ukrainian-catholic-university/hw2-ucu-2026
wan

In [7]:
def write_submission(preds, path):
    rows = []
    for iid, fn in zip(test_df.image_id, test_df.file_name):
        k = frame_key(fn)
        if k not in preds:
            rows.append((iid, "")); continue
        xyxy, s, c = preds[k]
        o = np.argsort(-s)[:PROTOCOL["max_det"]]
        rows.append((iid, " ".join(f"{int(c[i])} {s[i]:.4f} {xyxy[i, 0]:.1f} {xyxy[i, 1]:.1f} "
                                   f"{xyxy[i, 2] - xyxy[i, 0]:.1f} {xyxy[i, 3] - xyxy[i, 1]:.1f}" for i in o)))
    df = pd.DataFrame(rows, columns=["image_id", "PredictionString"])
    df.to_csv(path, index=False)
    chk = pd.read_csv(path, keep_default_na=False)
    assert len(chk) == len(test_df) and set(chk.image_id) == set(test_df.image_id)
    n = chk.PredictionString.str.split().str.len().fillna(0)
    assert (n % 6 == 0).all() and (n <= 6 * PROTOCOL["max_det"]).all()
    print(f"сабміт: {path} | кадрів з предиктами: {(n > 0).sum()} / {len(chk)}")


test_preds = predict_ultra(model, TEST, IMGSZ)
write_submission(test_preds, OUT / f"submission_{RUN_ID}.csv")

for tag, p in [("val", val_preds), ("test", test_preds)]:
    pickle.dump({k: (b, s.astype(np.float16), c) for k, (b, s, c) in p.items()}, open(OUT / f"{tag}_preds.pkl", "wb"))
json.dump(dict(run_id=RUN_ID, model=MODEL, change=CHANGE, imgsz=IMGSZ, epochs=EPOCHS, debug=DEBUG,
               protocol=PROTOCOL, train_kw=TRAIN_KW, ultralytics=ultralytics.__version__, **metrics),
          open(OUT / "metrics.json", "w"), indent=1)

row = [RUN_ID, FAMILY, MODEL.replace(".pt", ""), CHANGE, IMGSZ, EPOCHS] + \
      [f"{metrics[k]:.4f}" for k in ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]] + \
      [f"{metrics['FPS']:.1f}", "", f"{RUN_ID}_eval", "DEBUG" if DEBUG else ""]
print("\nрядок для code/results/experiments.csv:\n" + ",".join(map(str, row)))

сабміт: /kaggle/working/artifacts/A1_yolo11s_1280/submission_A1_yolo11s_1280.csv | кадрів з предиктами: 3811 / 3811

рядок для code/results/experiments.csv:
A1_yolo11s_1280,anchor_free,yolo11s,imgsz 1280,1280,10,0.1908,0.3787,0.1692,0.1110,0.1938,0.2892,48.4,,A1_yolo11s_1280_eval,


In [8]:
del model, val_preds, test_preds; gc.collect(); torch.cuda.empty_cache()
shutil.rmtree(run_dir, ignore_errors=True)
shutil.rmtree(WORK / "wandb", ignore_errors=True)
for f in WORK.glob("*.pt"):
    f.unlink()
print(sorted(p.name for p in OUT.iterdir()))

['args.yaml', 'best.pt', 'confusion_matrix_normalized.png', 'metrics.json', 'results.csv', 'results.png', 'submission_A1_yolo11s_1280.csv', 'test_preds.pkl', 'val_preds.pkl']
